<a href="https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Leakage Check

This assignment tests whether any model input directly reveals the outcome. The target below is an observed decline proxy, not a causal label.

Run the cells from top to bottom. The source dataset is the anonymized public starter sample; do not add private client data or credentials to this notebook.


## 1. Load the public anonymized sample

In [1]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_URL)
print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print(df.columns.tolist())

Rows: 30,000
Columns: 44
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 2. Define the target and identify suspicious features

In [2]:
target_candidates = [c for c in ["is_declining_label", "is_declining_proxy"] if c in df.columns]
if target_candidates:
    target = target_candidates[0]
elif "trend_direction" in df.columns:
    target = "trend_direction"
else:
    raise KeyError("Could not identify a decline target; inspect the displayed column list.")

forbidden_names = {"trend_direction", "trend_pct", "is_declining_label", "is_declining_proxy", "client_id", "client_hash_id", "content_id", "content_hash_id"}
suspicious = [c for c in df.columns if c in forbidden_names]
print("Chosen target:", target)
print("Fields requiring explicit exclusion or justification:", suspicious)

Chosen target: trend_direction
Fields requiring explicit exclusion or justification: ['content_id', 'client_id', 'trend_direction', 'trend_pct']


## 3. Check target construction and overlap

In [3]:
if target == "trend_direction":
    y = df[target].astype("string").str.lower().eq("down").astype(int)
else:
    y = pd.to_numeric(df[target], errors="coerce")
    if y.nunique(dropna=True) > 2:
        y = y.gt(y.median()).astype(int)
    else:
        y = y.fillna(0).astype(int)
print("Target rate:", round(float(y.mean()), 4))
print("Missing target rows:", int(y.isna().sum()))

candidate_features = [c for c in df.select_dtypes(include="number").columns if c not in forbidden_names]
print("Numeric candidate features:", candidate_features)
print("Forbidden target/identity fields excluded:", not (set(candidate_features) & forbidden_names))

Target rate: 0.5421
Missing target rows: 0
Numeric candidate features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
Forbidden target/identity fields excluded: True


## 4. Decision

**Interpretation:** Exclude the target, direct target ingredients, and identifiers from model inputs. Correlation alone does not prove leakage, but a feature used to construct the target is not a valid independent predictor. Document every feature whose timing or meaning could reveal future information.